# 00 - Setup, Environment, and Load Data

This notebook is the entry point for the challenge.

Purpose:

1. confirm the project root and imports
2. start Spark with the local project configuration
3. download and extract the Last.fm dataset automatically if it is not already present
4. stage the raw TSV into Parquet
5. load the normalized events table that all later notebooks use

Recommended next notebook: `01_data_eda_and_assumptions.ipynb`.


## Functions used in this notebook

### `create_spark_session(app_name, master='local[*]', shuffle_partitions=64)`
- **Description:** Creates the Spark session with the local Java/Hadoop helpers and scalable Spark defaults used across the project.
- **Input:** Application name and optional Spark execution settings.
- **Output:** A configured `SparkSession`.
- **Why this operation is selected for efficiency:** The configuration enables adaptive execution, tuned shuffle partitions, Snappy-compressed Parquet, and a safe local temp directory. That keeps the same notebook usable on larger datasets without rewriting setup logic.

### `ensure_lastfm_dataset_available(project_root, force_download=False, force_extract=False, verify_md5=True)`
- **Description:** Downloads the Last.fm archive from Zenodo if needed, verifies the archive checksum, and extracts the required TSV files into `data/raw/`.
- **Input:** Repository root path and optional refresh flags.
- **Output:** Absolute `Path` to the extracted raw event TSV.
- **Why this operation is selected for efficiency:** It removes manual setup for the reviewer while keeping the bootstrap idempotent: if the archive and extracted TSV already exist, nothing is downloaded again.

### `resolve_lastfm_archive_path(project_root)`
- **Description:** Returns the expected archive location inside `data/raw/`.
- **Input:** Repository root path.
- **Output:** Absolute `Path` to `lastfm-dataset-1K.tar.gz`.
- **Why this operation is selected for efficiency:** It makes the bootstrap paths explicit and keeps the notebook readable when printing where files were stored.

### `resolve_lastfm_parquet_path(project_root)`
- **Description:** Returns the expected Parquet staging directory.
- **Input:** Repository root path.
- **Output:** Absolute `Path` to the processed Parquet dataset.
- **Why this operation is selected for efficiency:** The first notebook can show both the raw and processed locations without recomputing path logic inline.

### `stage_lastfm_events_to_parquet(spark, project_root, overwrite=False, parquet_partitions=None)`
- **Description:** Reads the raw TSV once, normalizes the columns, repartitions by user, and writes a Parquet dataset.
- **Input:** Spark session, project root, and optional refresh controls.
- **Output:** Path to the staged Parquet dataset.
- **Why this operation is selected for efficiency:** TSV is fine as a raw delivery format, but Parquet is much faster for repeated analytical scans because it is columnar, compressed, and Spark-native.

### `load_lastfm_events(spark, project_root_or_input_path, prefer_parquet=True, refresh_parquet=False, parquet_partitions=None)`
- **Description:** Loads the normalized events table, preferably from the staged Parquet dataset.
- **Input:** Spark session plus either the project root or a direct input path.
- **Output:** Spark DataFrame of normalized play events.
- **Why this operation is selected for efficiency:** The notebooks can work from the raw TSV when needed, but the default path reuses Parquet so the heavy ingestion step is not repeated every time.



In [1]:
from pathlib import Path
import logging
import sys

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s:%(message)s')
logger = logging.getLogger('lastfm.notebook')

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


WindowsPath('C:/Users/GonzaloFigueroa/Documents/Private/BME/lastfm-analysis')

In [2]:
from pyspark.sql import functions as F

from src.spark_utils import create_spark_session
from src.sessionization import (
    ensure_lastfm_dataset_available,
    load_lastfm_events,
    resolve_lastfm_archive_path,
    resolve_lastfm_parquet_path,
    stage_lastfm_events_to_parquet,
)


In [3]:
spark = create_spark_session(app_name='00-setup-and-load-data')
spark


In [4]:
archive_path = resolve_lastfm_archive_path(PROJECT_ROOT)
raw_input_path = ensure_lastfm_dataset_available(PROJECT_ROOT)
parquet_path = resolve_lastfm_parquet_path(PROJECT_ROOT)

logger.info(f'Downloaded archive path: {archive_path}')
logger.info(f'Extracted raw input path: {raw_input_path}')
logger.info(f'Prepared Parquet path: {parquet_path}')


INFO:lastfm.notebook:Downloaded archive path: C:\Users\GonzaloFigueroa\Documents\Private\BME\lastfm-analysis\data\raw\lastfm-dataset-1K.tar.gz


INFO:lastfm.notebook:Extracted raw input path: C:\Users\GonzaloFigueroa\Documents\Private\BME\lastfm-analysis\data\raw\lastfm-dataset-1K\userid-timestamp-artid-artname-traid-traname.tsv


INFO:lastfm.notebook:Prepared Parquet path: C:\Users\GonzaloFigueroa\Documents\Private\BME\lastfm-analysis\data\processed\lastfm_events_parquet


In [5]:
parquet_path = stage_lastfm_events_to_parquet(spark, PROJECT_ROOT)
logger.info(f'Parquet dataset ready at: {parquet_path}')


INFO:lastfm.notebook:Parquet dataset ready at: C:\Users\GonzaloFigueroa\Documents\Private\BME\lastfm-analysis\data\processed\lastfm_events_parquet


In [6]:
events_df = load_lastfm_events(spark, PROJECT_ROOT)
events_df


DataFrame[user_id: string, started_at: timestamp, artist_id: string, artist_name: string, track_id: string, track_name: string, song_id: string]

In [7]:
events_df.printSchema()


root
 |-- user_id: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- song_id: string (nullable = true)



In [8]:
events_df.show(10, truncate=False)


+-----------+-------------------+------------------------------------+----------------------+------------------------------------+---------------------------+------------------------------------+
|user_id    |started_at         |artist_id                           |artist_name           |track_id                            |track_name                 |song_id                             |
+-----------+-------------------+------------------------------------+----------------------+------------------------------------+---------------------------+------------------------------------+
|user_000013|2006-12-22 18:07:11|835a6d9c-fea0-4a71-ae52-9c4da946433a|Styx                  |ea4bd0c5-c323-4d21-9c78-1bc9c46e5da8|Mr. Roboto                 |ea4bd0c5-c323-4d21-9c78-1bc9c46e5da8|
|user_000013|2006-12-22 18:12:00|NULL                                |Subwoofer             |NULL                                |Vaporize                   |Vaporize                            |
|user_000013|2006-12

In [9]:
events_df.agg(
    F.count('*').alias('total_rows'),
    F.countDistinct('user_id').alias('distinct_users'),
    F.min('started_at').alias('min_started_at'),
    F.max('started_at').alias('max_started_at'),
).show(truncate=False)


+----------+--------------+-------------------+-------------------+
|total_rows|distinct_users|min_started_at     |max_started_at     |
+----------+--------------+-------------------+-------------------+
|19150867  |992           |2005-02-14 00:00:07|2013-09-29 18:32:04|
+----------+--------------+-------------------+-------------------+



If the cells above succeed, the environment is ready.

Recommended next notebook: `01_data_eda_and_assumptions.ipynb`.


In [10]:
spark.stop()
